<a href="https://colab.research.google.com/github/sohan-paul/Qiskit-fall-fest-portfolio-optimization/blob/main/sp_intermidiate_challenge2_qiskit_portfolio_optimizatin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install qiskit qiskit-optimization


In [8]:
import numpy as np


In [9]:
import matplotlib.pyplot as plt


In [10]:
from qiskit_optimization.problems import QuadraticProgram

In [11]:
n=12

In [12]:
np.random.seed(0)
mu=np.random.uniform(0.005,0.015,n)

In [13]:
sigma=np.random.uniform(-0.05,0.5,(n,n))

In [14]:
sigma=np.dot(sigma,sigma.T)

In [15]:
q=1.5

In [16]:
k=6

In [17]:
import numpy as np
from qiskit_optimization.problems import QuadraticProgram
from qiskit_optimization.converters import QuadraticProgramToQubo

# Ensure required variables are defined
try:
    _ = n, mu, sigma, q, k
except NameError:
    n = 12
    np.random.seed(0)
    mu = np.random.uniform(0.005, 0.015, n)
    sigma = np.random.uniform(-0.05, 0.5, (n, n))
    sigma = np.dot(sigma, sigma.T)
    q = 1.5
    k = 6

qp = QuadraticProgram()

for i in range(n):
    qp.binary_var(name=f'x_{i}')

qp.minimize(linear=-(1 - q) * mu, quadratic=q * sigma)

# Add the budget constraint
qp.linear_constraint(linear=[1] * n, sense='==', rhs=k)
print("Original Model:\n", qp.export_as_lp_string())

# Convert the constrained problem into an unconstrained QUBO
qubo_converter = QuadraticProgramToQubo()
qubo_model = qubo_converter.convert(qp)

# Convert the unconstrained QUBO to an Ising Hamiltonian
qubit_op, offset = qubo_model.to_ising()

print("Successfully converted to Ising Hamiltonian!")

Original Model:
 \ This file has been generated by DOcplex
\ ENCODING=ISO-8859-1
\Problem name: CPLEX

Minimize
 obj: 0.005244067520 x_0 + 0.006075946832 x_1 + 0.005513816880 x_2
      + 0.005224415915 x_3 + 0.004618273997 x_4 + 0.005729470565 x_5
      + 0.004687936056 x_6 + 0.006958865004 x_7 + 0.007318313803 x_8
      + 0.004417207594 x_9 + 0.006458625190 x_10 + 0.005144474599 x_11 + [
      4.049044423301 x_0^2 + 4.224190639602 x_0*x_1 + 4.159320786187 x_0*x_2
      + 2.293775195822 x_0*x_3 + 3.626282859643 x_0*x_4 + 3.349324739153 x_0*x_5
      + 3.945111015852 x_0*x_6 + 4.315030870693 x_0*x_7 + 6.124291447830 x_0*x_8
      + 3.946404671806 x_0*x_9 + 5.096771099697 x_0*x_10
      + 5.615935525832 x_0*x_11 + 2.237443426502 x_1^2 + 3.018427802516 x_1*x_2
      + 2.311427242629 x_1*x_3 + 1.378458529973 x_1*x_4 + 1.813555772944 x_1*x_5
      + 2.533507314918 x_1*x_6 + 2.658439231276 x_1*x_7 + 4.694629237951 x_1*x_8
      + 2.985846288331 x_1*x_9 + 3.831346825235 x_1*x_10
      + 5.156

/tmp/ipykernel_511/1458379657.py:26: DeprecationWarning: The method ``qiskit_optimization.problems.quadratic_program.QuadraticProgram.export_as_lp_string()`` is deprecated as of Qiskit 0.7.0. It will be removed no earlier than 3 months after the release date. Use prettyprint instead.
  print("Original Model:\n", qp.export_as_lp_string())


In [18]:
!pip install qiskit-optimization

In [19]:
!pip install qiskit-algorithms

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 3.3 MB/s eta 0:00:00


In [ ]:
from qiskit_algorithms import NumPyMinimumEigensolver, QAOA
from qiskit_algorithms.optimizers import COBYLA
from qiskit_optimization.algorithms import MinimumEigenOptimizer, WarmStartQAOAOptimizer
from qiskit.primitives import StatevectorSampler
from qiskit.circuit import QuantumCircuit

# A simple, robust V2-to-V1 Sampler adapter wrapper to keep compatibility with qiskit-algorithms
class V1SamplerAdapter:
    def __init__(self):
        self._sampler = StatevectorSampler()

    def run(self, circuits, parameter_values=None, **run_options):
        # Standardize inputs into modern V2 Primitive Unified Blocks (PUBs)
        pubs = []

        # Handle different structures sent by different algorithms solvers
        if isinstance(circuits, list):
            for item in circuits:
                if isinstance(item, tuple):
                    # If already packaged as (circuit, parameters)
                    pubs.append(item)
                else:
                    pubs.append((item, parameter_values if parameter_values is not None else []))
        elif isinstance(circuits, tuple):
            # If it's a single tuple of (circuit, parameters)
            if len(circuits) == 2 and isinstance(circuits[0], QuantumCircuit):
                pubs.append(circuits)
            else:
                pubs.append((circuits, parameter_values if parameter_values is not None else []))
        else:
            pubs.append((circuits, parameter_values if parameter_values is not None else []))

        job = self._sampler.run(pubs)

        # Standardize results to resemble V1 expectations
        class V1JobMock:
            def __init__(self, v2_job):
                self.v2_job = v2_job
            def result(self):
                v2_result = self.v2_job.result()

                # Helper class to mock the data.meas hierarchical interface
                class MeasMock:
                    def __init__(self, pub_result):
                        self._pub_result = pub_result
                        self.num_shots = pub_result.data.meas.num_shots
                    def get_int_counts(self):
                        return self._pub_result.data.meas.get_int_counts()
                    def get_counts(self):
                        return self._pub_result.data.meas.get_counts()

                class DataMock:
                    def __init__(self, pub_result):
                        self.meas = MeasMock(pub_result)

                class PubResultMock:
                    def __init__(self, pub_result):
                        self.data = DataMock(pub_result)

                class V1ResultMock(list):
                    def __init__(self, r):
                        mocks = [PubResultMock(pub_res) for pub_res in r]
                        super().__init__(mocks)
                        self.quasi_dists = [pub_res.data.meas.get_counts() for pub_res in r]

                return V1ResultMock(v2_result)

        return V1JobMock(job)

# Set up the exact classical solver (pre-solver)
exact_solver = NumPyMinimumEigensolver()
pre_optimizer = MinimumEigenOptimizer(exact_solver)

# Set up the standard QAOA using our V1-adapted Sampler
qaoa = QAOA(sampler=V1SamplerAdapter(), optimizer=COBYLA(), reps=1)

# Set up the Warm Start QAOA Optimizer
optimizer = WarmStartQAOAOptimizer(
    pre_solver=pre_optimizer,
    relax_for_pre_solver=False,
    qaoa=qaoa
)

# Solve the unconstrained QUBO model using the warm-start strategy
result = optimizer.solve(qubo_model)

# Display the optimal portfolio selection
print("Optimal portfolio selection:", result.x)
print(f"Total number of stocks selected: {sum(result.x)} (Target: {k})")

/usr/local/lib/python3.13/dist-packages/scipy/sparse/linalg/_dsolve/linsolve.py:606: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/usr/local/lib/python3.13/dist-packages/scipy/sparse/linalg/_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
/usr/local/lib/python3.13/dist-packages/scipy/sparse/_index.py:168: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Calculate expected returns and risks for all assets
returns = mu
risks = np.sqrt(np.diag(sigma))

# Create scatter plot of risk vs return
plt.figure(figsize=(10, 6))
plt.scatter(risks, returns, color="blue", label="All Assets")

# Highlight selected stocks
selected_indices = np.where(result.x == 1)[0]
plt.scatter(
    risks[selected_indices],
    returns[selected_indices],
    color="red",
    s=100,
    label="Selected Portfolio",
)

# Add labels and title
plt.title("Portfolio Optimization: Risk vs. Expected Return")
plt.xlabel("Risk (Standard Deviation)")
plt.ylabel("Expected Return")
plt.axhline(0, color="grey", linewidth=0.5)
plt.axvline(0, color="grey", linewidth=0.5)
plt.grid(True, linestyle="--")
plt.legend()
plt.show()

In [ ]:
!pip install qiskit-aer

In [ ]:
import numpy as np

# Simple Greedy Algorithm: Select the K stocks with the highest expected return (mu)
# np.argsort sorts in ascending order, so we take the last 'k' elements for the highest values
top_k_indices = np.argsort(mu)[-k:][::-1]

# Create a binary array to match the format of your QAOA output
classical_greedy_selection = np.zeros(n)
classical_greedy_selection[top_k_indices] = 1

print("--- Classical Greedy Method ---")
print("Optimal portfolio selection:", classical_greedy_selection)
print(f"Total number of stocks selected: {sum(classical_greedy_selection)} (Target: {k})")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from qiskit_optimization.converters import LinearEqualityToPenalty

# NOTE: If your original constrained quadratic program variable is named
# something other than 'qp' (e.g., 'problem'), change it on line 12.

lambda_values = np.linspace(0.5, 10, 20) # Sweeping lambda from 0.5 to 10
fraction_valid = []

for lam in lambda_values:
    # 1. Convert the constrained problem to a QUBO using the current lambda penalty
    converter = LinearEqualityToPenalty(penalty=lam)
    qubo_lam = converter.convert(qp)

    # 2. Solve using your already defined WarmStartQAOAOptimizer
    result_lam = optimizer.solve(qubo_lam)

    # 3. Calculate the fraction (probability) of samples that buy exactly 'k' stocks
    valid_probability = 0
    for sample in result_lam.samples:
        if sum(sample.x) == k:
            valid_probability += sample.probability

    fraction_valid.append(valid_probability)

# 4. Plot the results
plt.figure(figsize=(10, 6))
plt.plot(lambda_values, fraction_valid, marker='o', color='purple')
plt.xlabel('Penalty Parameter ($\lambda$)')
plt.ylabel('Fraction of Valid Samples (Exactly K stocks)')
plt.title('Parameter Sweep: Fraction of Valid Portfolios vs $\lambda$')
plt.grid(True)
plt.show()